In [ ]:
from pathlib import Path
import pandas as pd
import os

data_path_2021 = os.environ.get("MALFRID_2021", None)

p = Path(data_path_2021)


# Initiell datautforsk

In [ ]:
info = sorted([e.name[:-6].split("_") + [e.name] for e in p.iterdir()])
df = pd.DataFrame(info, columns=["nettside", "språk", "format", "filnavn"])
df

In [ ]:
antall_filer, antall_unike_nettsider = len(df), len(list(df.groupby("nettside")))
fler_språk = 0
nynorsk = 0
nynorsk_flerspråk = 0
nynorsk_og_bokmål = 0

for nettside, df_ in df.groupby("nettside"):
    unike_språk = set(df_.språk)
    if len(unike_språk) > 1:
        fler_språk += 1
    if "nno" in unike_språk:
        nynorsk += 1
    if len(unike_språk) > 1 and "nno" in unike_språk:
        nynorsk_flerspråk += 1
    if "nno" in unike_språk and "nob" in unike_språk:
        nynorsk_og_bokmål += 1

In [ ]:
print(f"""
    Det er {antall_filer} filer i målfrid_data
    Det er {antall_unike_nettsider} unike nettsider
    Der er {fler_språk} nettsider som har fler språk 
    Der er {nynorsk} nettsider som har nynorsk 
    Det er {nynorsk_flerspråk} nettsider som har nynorsk og et annet språk
    Det er {nynorsk_og_bokmål} nettsider som har nynorsk og bokmål
""")

In [ ]:
from collections import defaultdict, Counter

df_path = Path("output/dokumenter_statistikk.csv")

if not df_path.exists():
    dok_data = defaultdict(Counter)
    for nettside, df_ in df.groupby("nettside"):    
        for språk, df__ in df_.groupby("språk"):
            filer = df__.filnavn
            for fil in filer:
                filp = p / fil
                jsonfil = pd.read_json(filp, lines=True)
                assert all(jsonfil["lang"] == språk)
                dok_data[nettside][språk] += len(jsonfil)

    dok_data_df = pd.DataFrame([(nettside, counter["nno"], counter["nob"]) for nettside, counter in dok_data.items()], columns=["nettside", "antall_nno_dokumenter", "antall_nob_dokumenter"])
    dok_data_df.to_csv(df_path, index=False)
else:
    dok_data_df = pd.read_csv("output/dokumenter_statistikk.csv")
dok_data_df

## Sidene med flest bokmåls-tekster

In [ ]:
dok_data_df.sort_values("antall_nob_dokumenter", ascending=False)[:30]

## Sidene med flest nynorsk-tekster

In [ ]:
dok_data_df.sort_values("antall_nno_dokumenter", ascending=False)[:30]

## Sidene med flest begge-tekster

In [ ]:
dok_data_df["sum_"]= dok_data_df.antall_nno_dokumenter + dok_data_df.antall_nob_dokumenter
dok_data_df.sort_values("sum_", ascending=False)[:20]

In [ ]:
def nn_bm_distribution(row):
    min_val = min(row.antall_nob_dokumenter, row.antall_nno_dokumenter)
    total = row.antall_nob_dokumenter + row.antall_nno_dokumenter
    return min_val/total


dok_data_df["nn_bm_distribution"] = dok_data_df.apply(nn_bm_distribution, axis=1)

dok_data_df[dok_data_df.sum_ > 200].sort_values("nn_bm_distribution", ascending=False)[:20]